# 📝 Agent 评估（Agent Evaluation）


**欢迎来到 Kaggle 5 天 Agents 课程第 4 天！**

在上一份 Notebook 中，我们探索了如何在 AI Agent 中实现可观测性（Observability）。这种方法主要是**被动式（reactive）**的：问题出现后，它会提供调试与定位根因所需的数据。

在本 Notebook 中，我们将用**Agent 评估（Agent Evaluation）**来补足这些可观测性实践，采用一种**主动式（proactive）**方法。通过持续评估 Agent 的表现，我们可以更早发现质量退化问题！

```
                            Observability + Agent Evaluation
                            (reactive)      (proactive)
```

## **什么是 Agent 评估（Agent Evaluation）？**

它是一个系统化过程，用于在不同场景和质量维度下测试并衡量 AI Agent 的表现。


## **🤖 场景故事**

你构建了一个家庭自动化 Agent。在你的测试中它运行得非常完美，于是你很有信心地上线了它……


* **第 1 周：** 🚨「我让 Agent 开灯，它却把壁炉打开了！」
* **第 2 周：** 🚨「Agent 在客房完全不响应指令！」
* **第 3 周：** 🚨「设备不可用时，Agent 的回复很不礼貌！」

**问题在于：** `常规测试 ≠ 评估`

Agent 与传统软件不同：
- 它们是非确定性的（non-deterministic）
- 用户会给出不可预测、含糊不清的指令
- 很小的 Prompt 改动也可能导致行为和工具调用发生巨大变化

为了应对这些差异，Agent 需要系统化评估，而不只是“happy path”测试。**这意味着你不仅要评估最终回复，还要评估它得出该回复所走的路径（trajectory）！**

完成本 Notebook 后，你将能够：

* ✅ 理解什么是 Agent 评估以及如何使用它
* ✅ 直接在 ADK Web UI 中运行评估并分析结果
* ✅ 识别 Agent 在一段时间内的性能回归（regression）
* ✅ 理解并创建必要的评估文件（`*.test.json`、`*.evalset.json`、`test_config.json`）。


---
## ⚙️ 第 1 部分：环境准备

在开始评估之旅前，我们先把环境配置好。

### 1.1：安装依赖

Kaggle Notebooks 环境已预装 Python 版 [google-adk](https://google.github.io/adk-docs/) 及其依赖，因此在本 Notebook 中你无需额外安装包。

如果你要在本课程之外、自己的 Python 开发环境中安装和使用 ADK，可以运行：

```
pip install google-adk
```

### 🔑 1.2：配置 Gemini API Key

本 Notebook 使用 [Gemini API](https://ai.google.dev/gemini-api/)，因此需要 API Key。

**1. 获取 API Key**

如果你还没有，请先在 [Google AI Studio 创建 API key](https://aistudio.google.com/app/api-keys)。

**2. 将 Key 添加到 Kaggle Secrets**

接下来，你需要把 API Key 作为 Kaggle User Secret 添加到 Notebook 中。

1. 在 Notebook 编辑器顶部菜单选择 `Add-ons`，然后点击 `Secrets`。
2. 创建一个新 secret，标签填写 `GOOGLE_API_KEY`。
3. 将 API Key 粘贴到 “Value” 字段，然后点击 “Save”。
4. 确保 `GOOGLE_API_KEY` 旁边的复选框已勾选，这样该 secret 才会挂载到 Notebook。

**3. 在 Notebook 中完成鉴权**

运行下面这个单元，读取你刚保存的 `GOOGLE_API_KEY` 并将其设置为环境变量供 Notebook 使用：

In [2]:
import os
from kaggle_secrets import UserSecretsClient

try:
    GOOGLE_API_KEY = UserSecretsClient().get_secret("GOOGLE_API_KEY")
    os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
    print("✅ Setup and authentication complete.")
except Exception as e:
    print(
        f"🔑 Authentication Error: Please make sure you have added 'GOOGLE_API_KEY' to your Kaggle secrets. Details: {e}"
    )

✅ Setup and authentication complete.


### 💻 1.3：设置代理与隧道

我们会通过代理在 Kaggle Notebooks 环境中访问 ADK Web UI。如果你是在 Kaggle 之外运行本 Notebook，则无需这一步。

In [3]:
from IPython.core.display import display, HTML
from jupyter_server.serverapp import list_running_servers


# Gets the proxied URL in the Kaggle Notebooks environment
def get_adk_proxy_url():
    PROXY_HOST = "https://kkb-production.jupyter-proxy.kaggle.net"
    ADK_PORT = "8000"

    servers = list(list_running_servers())
    if not servers:
        raise Exception("No running Jupyter servers found.")

    baseURL = servers[0]["base_url"]

    try:
        path_parts = baseURL.split("/")
        kernel = path_parts[2]
        token = path_parts[3]
    except IndexError:
        raise Exception(f"Could not parse kernel/token from base URL: {baseURL}")

    url_prefix = f"/k/{kernel}/{token}/proxy/proxy/{ADK_PORT}"
    url = f"{PROXY_HOST}{url_prefix}"

    styled_html = f"""
    <div style="padding: 15px; border: 2px solid #f0ad4e; border-radius: 8px; background-color: #fef9f0; margin: 20px 0;">
        <div style="font-family: sans-serif; margin-bottom: 12px; color: #333; font-size: 1.1em;">
            <strong>⚠️ IMPORTANT: Action Required</strong>
        </div>
        <div style="font-family: sans-serif; margin-bottom: 15px; color: #333; line-height: 1.5;">
            The ADK web UI is <strong>not running yet</strong>. You must start it in the next cell.
            <ol style="margin-top: 10px; padding-left: 20px;">
                <li style="margin-bottom: 5px;"><strong>Run the next cell</strong> (the one with <code>!adk web ...</code>) to start the ADK web UI.</li>
                <li style="margin-bottom: 5px;">Wait for that cell to show it is "Running" (it will not "complete").</li>
                <li>Once it's running, <strong>return to this button</strong> and click it to open the UI.</li>
            </ol>
            <em style="font-size: 0.9em; color: #555;">(If you click the button before running the next cell, you will get a 500 error.)</em>
        </div>
        <a href='{url}' target='_blank' style="
            display: inline-block; background-color: #1a73e8; color: white; padding: 10px 20px;
            text-decoration: none; border-radius: 25px; font-family: sans-serif; font-weight: 500;
            box-shadow: 0 2px 5px rgba(0,0,0,0.2); transition: all 0.2s ease;">
            Open ADK Web UI (after running cell below) ↗
        </a>
    </div>
    """

    display(HTML(styled_html))

    return url_prefix


print("✅ Helper functions defined.")

✅ Helper functions defined.


---
## 🏠 第 2 部分：创建家庭自动化 Agent

现在来创建一个将作为评估主角的 Agent。这个家庭自动化 Agent 在基础测试里看起来很完美，但它隐藏的问题会在全面评估中暴露出来。先运行 `adk create` CLI 命令创建项目脚手架。

In [4]:
!adk create home_automation_agent --model gemini-2.5-flash-lite --api_key $GOOGLE_API_KEY


Agent created in /kaggle/working/home_automation_agent:
- .env
- __init__.py
- agent.py



运行下面单元来创建家庭自动化 Agent。

该 Agent 使用一个 `set_device_status` 工具来控制智能家居设备。设备状态只能是 ON 或 OFF。**它的指令被刻意设置得过于自信**：声称可以控制“所有智能设备”和“用户提到的任意设备”，这正是后面评估会发现问题的伏笔。

In [5]:
%%writefile home_automation_agent/agent.py

from google.adk.agents import LlmAgent
from google.adk.models.google_llm import Gemini

from google.genai import types

# Configure Model Retry on errors
retry_config = types.HttpRetryOptions(
    attempts=5,  # Maximum retry attempts
    exp_base=7,  # Delay multiplier
    initial_delay=1,
    http_status_codes=[429, 500, 503, 504],  # Retry on these HTTP errors
)

def set_device_status(location: str, device_id: str, status: str) -> dict:
    """Sets the status of a smart home device.

    Args:
        location: The room where the device is located.
        device_id: The unique identifier for the device.
        status: The desired status, either 'ON' or 'OFF'.

    Returns:
        A dictionary confirming the action.
    """
    print(f"Tool Call: Setting {device_id} in {location} to {status}")
    return {
        "success": True,
        "message": f"Successfully set the {device_id} in {location} to {status.lower()}."
    }

# This agent has DELIBERATE FLAWS that we'll discover through evaluation!
root_agent = LlmAgent(
    model=Gemini(model="gemini-2.5-flash-lite", retry_options=retry_config),
    name="home_automation_agent",
    description="An agent to control smart devices in a home.",
    instruction="""You are a home automation assistant. You control ALL smart devices in the house.
    
    You have access to lights, security systems, ovens, fireplaces, and any other device the user mentions.
    Always try to be helpful and control whatever device the user asks for.
    
    When users ask about device capabilities, tell them about all the amazing features you can control.""",
    tools=[set_device_status],
)

Overwriting home_automation_agent/agent.py


---
## ✔️ 第 3 部分：在 ADK Web UI 中进行交互式评估

### 3.1：启动 ADK Web UI

先获取在 Kaggle Notebooks 环境中访问 ADK Web UI 所需的代理 URL：

In [6]:
url_prefix = get_adk_proxy_url()

现在你可以通过下面命令启动 ADK Web UI。

👉 **注意：** 下一个单元不会“执行结束”，它会持续运行并提供 ADK Web UI 服务，直到你手动停止该单元。

In [7]:
!adk web --url_prefix {url_prefix}

/usr/local/lib/python3.11/dist-packages/google/adk/cli/fast_api.py:130: UserWarning: [EXPERIMENTAL] InMemoryCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  credential_service = InMemoryCredentialService()
/usr/local/lib/python3.11/dist-packages/google/adk/auth/credential_service/in_memory_credential_service.py:33: UserWarning: [EXPERIMENTAL] BaseCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  super().__init__()
INFO:     Started server process [87]
INFO:     Waiting for application startup.

+-----------------------------------------------------------------------------+
| ADK Web Server started                                                      |
|                                                                             |
| For local testing, access at http:/

当 ADK Web UI 启动后，使用前一个单元中的按钮打开代理链接。

‼️ **重要：不要把这个代理链接分享给任何人**，因为 URL 中包含你的认证 token，属于敏感数据。

### 3.2：创建你的第一个“完美”测试用例

**👉 操作：在 ADK Web UI 中**

1. 点击上方公共 URL 打开 ADK Web UI
2. 从下拉框选择 `home_automation_agent`
3. **进行一次正常对话：** 输入 `Turn on the desk lamp in the office`
4. **Agent 正确响应**：控制设备并确认动作

**👉 操作：将其保存为第一个评估用例**

1. 在右侧面板进入 **Eval** 标签页
2. 点击 **Create Evaluation set**，命名为 `home_automation_tests`
3. 在 `home_automation_tests` 中点击 `>` 箭头，然后点 **Add current session**
4. 将该用例命名为 `basic_device_control`

**✅ 成功！** 你刚刚把第一次交互保存成了一个评估用例。

![Create Test Cases](https://storage.googleapis.com/github-repo/kaggle-5days-ai/day4/eval-create-testcase.gif)

### 3.3：运行评估

**👉 操作：运行你的第一次评估**

现在运行这个测试用例，看看 Agent 能否复现它之前的成功表现。

1. 在 Eval 标签页中，确认你的新测试用例已勾选。
2. 点击 Run Evaluation 按钮。
3. 会弹出 EVALUATION METRIC 对话框。先保持默认值并点击 Start。
4. 评估运行后，你应在 Evaluation History 看到绿色 Pass 结果。这表示 Agent 的行为与保存会话一致。

‼️ **理解评估指标**

运行评估时，你会看到两个关键分数：

* **Response Match Score：** 衡量 Agent 实际回复与期望回复的相似度。使用文本相似度算法比较内容。1.0 表示完全匹配，0.0 表示完全不同。

* **Tool Trajectory Score：** 衡量 Agent 是否以正确参数调用了正确工具。它会检查工具调用序列是否符合预期。1.0 表示工具使用完全正确，0.0 表示工具或参数错误。

**👉 操作：分析一次失败**

我们故意破坏这个测试，看看失败时长什么样。

1. 在 eval 用例列表中，点击目标用例旁边的 Edit（铅笔）图标。
2. 在 “Final Response” 文本框中，把期望文本改成错误内容，例如：`The desk lamp is off`。
3. 保存修改后重新运行评估。
4. 这次结果会变成红色 Fail。将鼠标悬停在 “Fail” 标签上，会出现提示框并并排展示 Actual 与 Expected Output，明确指出失败原因（最终回复不匹配）。
这种即时且细粒度的反馈对调试非常有价值。

![Evaluate](https://storage.googleapis.com/github-repo/kaggle-5days-ai/day4/eval-run-test.gif)

### 3.4：（可选）创建更有挑战的测试用例

现在创建更多测试用例来暴露隐藏问题：

**在不同对话中创建以下场景：**

1. **歧义指令：** `"Turn on the lights in the bedroom"`
   - 另存为新测试用例：`ambiguous_device_reference`
   - 运行评估：它可能通过，但 Agent 也许存在理解偏差

2. **无效位置：** `"Please turn off the TV in the garage"`  
   - 另存为新测试用例：`invalid_location_test`
   - 运行评估：Agent 可能尝试控制不存在的设备

3. **复杂指令：** `"Turn off all lights and turn on security system"`
   - 另存为新测试用例：`complex_multi_device_command`
   - 运行评估：Agent 可能尝试执行超出能力范围的操作

**你将发现的问题：**
即使测试显示“通过”，你仍可能观察到 Agent：
- 对不存在的设备做出假设
- 给出看似有帮助但并不准确的回复
- 试图控制本不应具备权限的设备

## 🤔 我还缺了什么？

❌ **Web UI 的局限：** 到目前为止，我们看到了如何在 ADK Web UI 中创建并评估测试用例。Web UI 很适合交互式创建测试，但一次只测一段对话，无法规模化。

❓ **关键问题：** 我该如何主动检测 Agent 性能回归？

下一部分就来回答这个问题！

---

## ‼️ **停止 ADK Web UI** 🛑

**为了运行本 Notebook 剩余部分的单元，** 请先停止你在 3.1 中启动 `adk web` 的那个运行中单元。

否则只要 ADK Web UI 持续运行，该单元就会阻塞并阻止其他单元执行。

---
## 📈 第 4 部分：系统化评估

回归测试（Regression Testing）指的是重新运行已有测试，以确保新改动没有破坏此前可用的功能。

ADK 提供两种自动回归与批量测试方式：[pytest](https://google.github.io/adk-docs/evaluate/#2-pytest-run-tests-programmatically) 和 [adk eval](https://google.github.io/adk-docs/evaluate/#3-adk-eval-run-evaluations-via-the-cli) CLI 命令。本节我们使用 CLI 命令。若想了解 `pytest` 方式，请查看本 Notebook 末尾资源链接。

下图展示了评估的整体流程。**高层上，评估分为四步：**

1) **创建评估配置**：定义指标，也就是你要衡量什么
2) **创建测试用例**：准备用于对比的样例测试
3) **让 Agent 处理测试查询**
4) **对比结果**



![Evaluate](https://storage.googleapis.com/github-repo/kaggle-5days-ai/day4/evaluate_agent.png)

### 4.1：创建评估配置

这个可选文件允许我们定义通过/失败阈值。请在根目录创建 `test_config.json`。

In [8]:
import json

# Create evaluation configuration with basic criteria
eval_config = {
    "criteria": {
        "tool_trajectory_avg_score": 1.0,  # Perfect tool usage required
        "response_match_score": 0.8,  # 80% text similarity threshold
    }
}

with open("home_automation_agent/test_config.json", "w") as f:
    json.dump(eval_config, f, indent=2)

print("✅ Evaluation configuration created!")
print("\n📊 Evaluation Criteria:")
print("• tool_trajectory_avg_score: 1.0 - Requires exact tool usage match")
print("• response_match_score: 0.8 - Requires 80% text similarity")
print("\n🎯 What this evaluation will catch:")
print("✅ Incorrect tool usage (wrong device, location, or status)")
print("✅ Poor response quality and communication")
print("✅ Deviations from expected behavior patterns")

✅ Evaluation configuration created!

📊 Evaluation Criteria:
• tool_trajectory_avg_score: 1.0 - Requires exact tool usage match
• response_match_score: 0.8 - Requires 80% text similarity

🎯 What this evaluation will catch:
✅ Incorrect tool usage (wrong device, location, or status)
✅ Poor response quality and communication
✅ Deviations from expected behavior patterns


### 4.2：创建测试用例

这个文件（`integration.evalset.json`）将包含多个测试用例（sessions）。

这个评估集既可以通过合成方式创建，也可以来源于 ADK Web UI 中的会话。

**提示：** 若要持久化 ADK Web UI 中的对话，只需在 UI 里创建一个 evalset，并把当前 session 加入其中。该 session 内的所有对话会自动转换为 evalset 并下载到本地。

In [13]:
# Create evaluation test cases that reveal tool usage and response quality problems
test_cases = {
    "eval_set_id": "home_automation_integration_suite",
    "eval_cases": [
        {
            "eval_id": "living_room_light_on",
            "conversation": [
                {
                    "user_content": {
                        "parts": [
                            {"text": "Please turn on the floor lamp in the living room"}
                        ]
                    },
                    "final_response": {
                        "parts": [
                            {
                                "text": "Successfully set the floor lamp in the living room to on."
                            }
                        ]
                    },
                    "intermediate_data": {
                        "tool_uses": [
                            {
                                "name": "set_device_status",
                                "args": {
                                    "location": "living room",
                                    "device_id": "floor lamp",
                                    "status": "ON",
                                },
                            }
                        ]
                    },
                }
            ],
        },
        {
            "eval_id": "kitchen_on_off_sequence",
            "conversation": [
                {
                    "user_content": {
                        "parts": [{"text": "Switch on the main light in the kitchen."}]
                    },
                    "final_response": {
                        "parts": [
                            {
                                "text": "Successfully set the main light in the kitchen to on."
                            }
                        ]
                    },
                    "intermediate_data": {
                        "tool_uses": [
                            {
                                "name": "set_device_status",
                                "args": {
                                    "location": "kitchen",
                                    "device_id": "main light",
                                    "status": "ON",
                                },
                            }
                        ]
                    },
                }
            ],
        },
    ],
}

现在把这些测试用例写入 Agent 根目录下的 `integration.evalset.json`。

In [10]:
import json

with open("home_automation_agent/integration.evalset.json", "w") as f:
    json.dump(test_cases, f, indent=2)

print("✅ Evaluation test cases created")
print("\n🧪 Test scenarios:")
for case in test_cases["eval_cases"]:
    user_msg = case["conversation"][0]["user_content"]["parts"][0]["text"]
    print(f"• {case['eval_id']}: {user_msg}")

print("\n📊 Expected results:")
print("• basic_device_control: Should pass both criteria")
print(
    "• wrong_tool_usage_test: May fail tool_trajectory if agent uses wrong parameters"
)
print(
    "• poor_response_quality_test: May fail response_match if response differs too much"
)

✅ Evaluation test cases created

🧪 Test scenarios:
• living_room_light_on: Please turn on the floor lamp in the living room
• kitchen_on_off_sequence: Switch on the main light in the kitchen.

📊 Expected results:
• basic_device_control: Should pass both criteria
• wrong_tool_usage_test: May fail tool_trajectory if agent uses wrong parameters
• poor_response_quality_test: May fail response_match if response differs too much


### 4.3：运行 CLI 评估

执行 `adk eval` 命令，并传入 Agent 目录、evalset 文件以及配置文件。

In [11]:
print("🚀 Run this command to execute evaluation:")
!adk eval home_automation_agent home_automation_agent/integration.evalset.json --config_file_path=home_automation_agent/test_config.json --print_detailed_results

🚀 Run this command to execute evaluation:
^C
Traceback (most recent call last):
  File "/usr/local/bin/adk", line 4, in <module>
    from google.adk.cli import main
  File "/usr/local/lib/python3.11/dist-packages/google/adk/__init__.py", line 16, in <module>
    from .agents.llm_agent import Agent
  File "/usr/local/lib/python3.11/dist-packages/google/adk/agents/__init__.py", line 18, in <module>
    from .base_agent import BaseAgent
  File "/usr/local/lib/python3.11/dist-packages/google/adk/agents/base_agent.py", line 40, in <module>
    from ..events.event import Event
  File "/usr/local/lib/python3.11/dist-packages/google/adk/events/__init__.py", line 15, in <module>
    from .event import Event
  File "/usr/local/lib/python3.11/dist-packages/google/adk/events/event.py", line 26, in <module>
    from ..models.llm_response import LlmResponse
  File "/usr/local/lib/python3.11/dist-packages/google/adk/models/__init__.py", line 19, in <module>
    from .gemma_llm import Gemma
  File "/u

### 4.4：分析评估结果示例

该命令会运行所有测试用例并输出汇总。`--print_detailed_results` 参数会给出逐轮明细，显示每个测试的分数，以及失败项的 diff 对比。


In [14]:
# Analyzing evaluation results - the data science approach
print("📊 Understanding Evaluation Results:")
print()
print("🔍 EXAMPLE ANALYSIS:")
print()
print("Test Case: living_room_light_on")
print("  ❌ response_match_score: 0.45/0.80")
print("  ✅ tool_trajectory_avg_score: 1.0/1.0")
print()
print("📈 What this tells us:")
print("• TOOL USAGE: Perfect - Agent used correct tool with correct parameters")
print("• RESPONSE QUALITY: Poor - Response text too different from expected")
print("• ROOT CAUSE: Agent's communication style, not functionality")
print()
print("🎯 ACTIONABLE INSIGHTS:")
print("1. Technical capability works (tool usage perfect)")
print("2. Communication needs improvement (response quality failed)")
print("3. Fix: Update agent instructions for clearer language or constrained response.")
print()

📊 Understanding Evaluation Results:

🔍 EXAMPLE ANALYSIS:

Test Case: living_room_light_on
  ❌ response_match_score: 0.45/0.80
  ✅ tool_trajectory_avg_score: 1.0/1.0

📈 What this tells us:
• TOOL USAGE: Perfect - Agent used correct tool with correct parameters
• RESPONSE QUALITY: Poor - Response text too different from expected
• ROOT CAUSE: Agent's communication style, not functionality

🎯 ACTIONABLE INSIGHTS:
1. Technical capability works (tool usage perfect)
2. Communication needs improvement (response quality failed)
3. Fix: Update agent instructions for clearer language or constrained response.



---
## 📚 第 5 部分：用户模拟（可选）

虽然**传统评估方法依赖固定测试用例**，但真实世界中的对话是动态且不可预测的。这正是用户模拟（User Simulation）的价值所在。

User Simulation 是 ADK 中一个强大的能力，用来弥补静态评估的局限。它不依赖预定义、固定的用户提示，而是在评估过程中使用生成式 AI 模型（如 Gemini）**动态生成用户提示。**

### ❓ 工作原理

* 你先定义一个 `ConversationScenario`，描述用户整体对话目标，并提供一个 `conversation_plan` 作为对话引导。
* 然后由大语言模型（LLM）扮演模拟用户，基于该计划和当前对话历史生成真实且多样的提示。
* 这使你能够更全面地测试 Agent：是否能处理意外转折、保持上下文，并在更自然且不可预测的流程中达成复杂目标。

User Simulation 能帮助你发现静态测试常常遗漏的边界场景，并提升 Agent 的鲁棒性。

### 👉 练习

现在你已经了解了 User Simulation 在动态 Agent 评估中的价值，下面是实践练习：

将**User Simulation**能力应用到你的 Agent。为某个具体目标定义一个包含 `conversation_plan` 的 `ConversationScenario`，并把它集成进 Agent 的评估流程。

**⭐ 参考这份[文档](https://google.github.io/adk-docs/evaluate/user-sim/)了解具体做法。**

## 🏆 恭喜！

### 你已经学会了

- ✅ 在 ADK Web UI 中进行交互式测试创建与分析
- ✅ 理解工具轨迹（Tool Trajectory）与回复质量（Response）指标
- ✅ 使用 `adk eval` CLI 命令进行自动化回归测试
- ✅ 如何分析评估结果并据此修复 Agent

**ℹ️ 说明：无需提交！**

本 Notebook 仅用于动手练习和学习。你**不需要**将它提交到任何地方即可完成课程。

### 📚 资源
* [ADK Evaluation 概览](https://google.github.io/adk-docs/evaluate/)
* 不同的[评估标准](https://google.github.io/adk-docs/evaluate/criteria/)
* [基于 Pytest 的评估](https://google.github.io/adk-docs/evaluate/#2-pytest-run-tests-programmatically)

### 进阶评估
对于生产部署，ADK 支持如 `safety_v1` 与 `hallucinations_v1` 等[高级评估标准](https://docs.cloud.google.com/vertex-ai/generative-ai/docs/models/determine-eval)（需要 Google Cloud 凭据）。

### 🎯 下一步
准备好迎接下一项挑战了吗？敬请期待最后的 Day 5 Notebook，我们将把这些能力真正串起来！

我们会学习如何**将 Agent 部署到生产环境**，并通过 **Agent2Agent Protocol** 扩展能力。